# Autograd from Scratch — Complete Beginner's Guide

This notebook builds a working automatic differentiation engine from zero.

**You will:**
1. Learn what derivatives, gradients, and computation graphs are
2. Build the engine line by line — understanding every character
3. Build a small neural network on top
4. Train it to learn XOR

**How to use this notebook:**
- Read every markdown cell completely before touching the code cell below it
- Type the code yourself — do not copy-paste
- If you get stuck, `autograd.ipynb` in the same folder has the finished code

No prior machine learning knowledge assumed. Basic Python required.

## What are we actually building?

A **neural network** learns by adjusting thousands of numbers (called **weights**)
until its outputs match the desired outputs. To adjust them correctly it needs to know:

> "If I nudge this weight up by a tiny amount, does the error go up or down — and by how much?"

That "by how much" is called the **gradient**. Computing gradients for thousands of
weights by hand is impossible. **Autograd** (automatic differentiation) does it automatically.

The idea: instead of working with plain numbers, we wrap every number in a special object
that remembers how it was computed. Once we have the full computation recorded, we walk
it backwards and apply one rule — the **chain rule** — to find every gradient at once.

That is the entire engine. Let's build it.

## What is a derivative? What is a gradient?

### Derivative (one variable)

A derivative measures **rate of change**.

If `y = x²`, then at `x = 3`, a tiny nudge `+ε` to `x` changes `y` by roughly `6ε`.
We write `dy/dx = 6` at that point. (In general, `d(x²)/dx = 2x`.)

Think of it as: *"for every 1 unit I increase x, y increases by 6 units."*

### Gradient (many variables)

When a function has many inputs — like `L(w1, w2, w3, ...)` — the **gradient** is just
the collection of all partial derivatives:

```
∇L = [ ∂L/∂w1,  ∂L/∂w2,  ∂L/∂w3, ... ]
```

Each entry answers: *"if I nudge just this one weight, how much does L change?"*

### Why this matters for neural networks

`L` is the **loss** — a number measuring how wrong the model is.
We want `L` to be as small as possible.
The gradient of `L` with respect to every weight tells us which direction each weight
should move to make `L` smaller. That is gradient descent.

### What `.grad` means in our code

When you see `v.grad`, it means `∂L/∂v` — how much the final loss `L` would change
if `v` increased by one tiny unit. We will compute this for every single `Value` in the graph.

## What is a computation graph?

When Python evaluates `L = (a * b + c) * f`, it produces one number and forgets everything.

A **computation graph** is a flowchart of that calculation — every intermediate result
kept as a node, every operation kept as a labelled edge.

```
a ──┐
    * ── e ──┐
b ──┘        + ── d ──┐
             │        * ── L
c ───────────┘        │
                 f ───┘
```

**Forward pass** — evaluate left to right, filling in the data at each node.

**Backward pass** — walk right to left, applying the chain rule at each node to fill
in the gradient at each node.

Our `Value` class builds this graph automatically just by doing arithmetic.
Every time you write `a + b`, a new node appears in the graph with `a` and `b` recorded
as its parents. Nothing special to do — it happens for free.

## Step 1 — Imports

Type these 3 lines in the cell below:

```python
import math
import random
from graphviz import Digraph
```

**What each one does:**

`import math` — gives us `math.tanh()`, `math.exp()`, `math.log()`.
Python's built-in `+`, `*` handle plain numbers but not these special functions.

`import random` — we will use `random.uniform(-1, 1)` to initialise neural network
weights with small random values. If all weights start at the same value, all neurons
learn the same thing and the network never improves.

`from graphviz import Digraph` — `Digraph` draws directed graphs (boxes connected by arrows).
We will use it to visualise our computation graph so you can see gradients flowing backwards.

In [ ]:
# ✏️  type your code here


## What does the `Value` class need to store?

Before writing any code, let's decide what fields each `Value` node needs.

| Field | Type | Purpose |
|-------|------|---------|
| `data` | `float` | The actual number at this node |
| `grad` | `float` | ∂L/∂self — filled in during backward pass |
| `_prev` | `set` | The parent `Value` nodes that produced this one |
| `_op` | `str` | Which operation produced this node (`'+'`, `'*'`, `'tanh'`, …) |
| `_backward` | function | The closure that propagates grad one step backwards |
| `label` | `str` | Optional human-readable name for visualisation |

`_prev` and `_op` record the **history** of how each node was produced.
`grad` and `_backward` are used during the **backward pass** to propagate gradients.

We start by writing the constructor (`__init__`) that initialises all of these.

## Step 2 — `Value.__init__`

Type this:

```python
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data      = float(data)
        self.grad      = 0.0
        self._backward = lambda: None
        self._prev     = set(_children)
        self._op       = _op
        self.label     = label
```

**Every line explained:**

`class Value:` — defines a new Python class. A class is a blueprint for creating objects.
Writing `Value(2.0)` uses this blueprint to create one instance.

`def __init__(self, data, _children=(), _op='', label=''):` — the constructor.
Python calls it automatically whenever you write `Value(...)`.
- `self` — always the first parameter of any method; refers to the object being created
- `data` — the number to wrap, e.g. `2.0`
- `_children=()` — tuple of `Value` objects that produced this one. Default: empty `()`
  because input values like `x = Value(2.0)` have no parents
- `_op=''` — operation string. Empty `''` for inputs, `'+'` for sums, etc.
- `label=''` — optional name, only used for drawing the graph

`self.data = float(data)` — `float()` converts e.g. `2` (int) to `2.0` (float).
This prevents bugs like `2 / 3 = 0` from integer division.

`self.grad = 0.0` — starts at zero. We have not run backprop yet, so we do not know the gradient.

`self._backward = lambda: None` — a function that does nothing.
`lambda: None` means "a function with no arguments that returns None".
This is a placeholder. When we define `__add__`, `__mul__`, etc., each one will
*replace* this placeholder with a real gradient function.

`self._prev = set(_children)` — converts the tuple of parents to a Python `set`.
Sets have O(1) membership check (`v in nodes`) which matters when walking large graphs.

`self._op = _op` and `self.label = label` — store as-is.

In [ ]:
# ✏️  type your code here


## Step 3 — `__repr__`: controlling how a `Value` prints

Add this method **inside** the `Value` class (indented 4 spaces):

```python
    def __repr__(self):
        return f'Value(data={self.data:.4f}, grad={self.grad:.4f})'
```

**What is `__repr__`?**
Python calls `__repr__` when you `print()` an object or type its name in a Jupyter cell.
Without it you get something like `<__main__.Value object at 0x10f3a2b90>` — not useful.

`f'...'` is an **f-string**. Anything inside `{}` gets evaluated as Python.
`:.4f` means: format as a **f**loat with **4** decimal places.

**Test it:** after typing the updated class, run:
```python
x = Value(3.14159)
print(x)   # should show: Value(data=3.1416, grad=0.0000)
```

In [ ]:
# ✏️  type your code here


## What should `a + b` do when `a` and `b` are `Value` objects?

Two things:

1. **Compute the result** — `a.data + b.data` — same as normal arithmetic
2. **Record the history** — create a new `Value` whose `_prev` contains `a` and `b`,
   and whose `_op` is `'+'`

This is how the computation graph grows: every operation adds a new node that points
back to its inputs.

Later, when we call `backward()`, we will walk these `_prev` links backwards from
the output all the way to the inputs, computing gradients at every step.

The operation methods (`__add__`, `__mul__`, etc.) are called **dunder methods**
(double-underscore). Python calls them automatically:
- `a + b` calls `a.__add__(b)`
- `a * b` calls `a.__mul__(b)`
- `a ** 2` calls `a.__pow__(2)`

## Step 4 — `__add__` and `__mul__`

Add these two methods inside `Value`:

```python
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), '+')

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), '*')
```

**`__add__` line by line:**

`other = other if isinstance(other, Value) else Value(other)` —
`isinstance(other, Value)` checks whether `other` is already a `Value` object.
If it is, use it directly. If it is a plain number like `2.0`, wrap it in `Value(2.0)`.
This makes `v + 2.0` work, not just `v + Value(2.0)`.

`return Value(self.data + other.data, (self, other), '+')` —
create a new `Value`. The three arguments are:
- `self.data + other.data` → the result number (the `data` field)
- `(self, other)` → the parents of this new node (stored as `_prev`)
- `'+'` → the operation label (stored as `_op`)

**`__mul__`** is identical in structure — just multiply instead of add.

**The key insight:** we return a *new* `Value` every time, and that new `Value` records
who its parents are. This is how the computation graph is built automatically —
just by doing arithmetic.

In [ ]:
# ✏️  type your code here


## Step 5 — `__pow__`, `tanh`, `relu`, `exp`, `log`

Add these inside `Value`:

```python
    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        return Value(self.data ** exp, (self,), f'**{exp}')

    def tanh(self):
        return Value(math.tanh(self.data), (self,), 'tanh')

    def relu(self):
        return Value(max(0.0, self.data), (self,), 'relu')

    def exp(self):
        return Value(math.exp(self.data), (self,), 'exp')

    def log(self):
        assert self.data > 0, f'log({self.data}) is undefined'
        return Value(math.log(self.data), (self,), 'log')
```

**What each function does mathematically:**

`tanh(x)` — "hyperbolic tangent". Squashes any number into the range (-1, +1).
Input 0 → output 0. Large positive → near +1. Large negative → near -1.
Used as an activation function in neurons to introduce non-linearity.

`relu(x)` — "Rectified Linear Unit". Simply `max(0, x)`. Negative → 0, positive → unchanged.
The most common activation in modern deep learning.

`exp(x)` — `eˣ` where `e ≈ 2.718`. Grows extremely fast. Used inside softmax, etc.

`log(x)` — natural logarithm. Undefined for `x ≤ 0` — hence the `assert`.

**Why `(self,)` and not `(self, other)`?**
These are **unary** operations — one input, one output. The exponent in `__pow__` is
a plain constant, not a tracked `Value`, so it does not appear in `_prev`.

In [ ]:
# ✏️  type your code here


## Step 6 — Convenience methods

Add these at the bottom of `Value`:

```python
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __neg__(self):         return self * -1
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other):  return self * other**-1
    def __rtruediv__(self, other): return other * self**-1
```

**Why do we need `__radd__` and `__rmul__`?**

When Python sees `2.0 + v` (a number plus a `Value`), it first tries `(2.0).__add__(v)`.
Python's built-in `float` does not know how to handle a `Value`, so it returns `NotImplemented`.
Python then tries the *right-hand* version: `v.__radd__(2.0)`. We just flip the order.

**Why no new math for subtract and divide?**

`a - b` is `a + (-b)`. So we use `__add__` and `__neg__`.
`a / b` is `a × b⁻¹`. So we use `__mul__` and `__pow__`.

This is intentional: **every complex operation is built from simpler ones we already defined.**
This means their backward passes are also automatically correct — we get subtraction
and division gradients for free without writing a single new `_backward` function.

In [ ]:
# ✏️  type your code here


## Step 7 — Assemble and test the forward pass

In the cell below, type the **complete `Value` class** — everything from Steps 2–6
together. Then test it:

```python
a = Value(2.0,  label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
f = Value(-2.0, label='f')

e = a * b;  e.label = 'e'
d = e + c;  d.label = 'd'
L = d * f;  L.label = 'L'

print(L)
print('op:     ', L._op)
print('parents:', L._prev)
```

**Work out the answer by hand first:**
- `e = a * b = 2 × -3 = -6`
- `d = e + c = -6 + 10 = 4`
- `L = d * f = 4 × -2 = -8`

So `L.data` should be `-8.0`. `L.grad` is `0.0` — backprop has not run yet.
`L._op` should be `'*'`. `L._prev` should contain `d` and `f`.

In [ ]:
# ✏️  type your code here


## Why visualise the computation graph?

Once you add backprop (Step 13+), each node will have both a `data` value and a `grad` value.

The visualisation lets you:
1. **Verify the graph shape** — did the operations connect correctly?
2. **Debug gradients** — after `backward()`, you can see the gradient at every node
   and check if they make sense (or find where they went wrong)

Without this, debugging autograd is nearly impossible.
PyTorch has `torchviz` for the same purpose. We are building our own.

## Step 8 — `trace` and `draw_dot`

Write these two functions **outside** the `Value` class (no indentation).

### `trace(root)` — collects all nodes and edges by walking the graph

```python
def trace(root):
    nodes, edges = set(), set()
    def walk(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                walk(child)
    walk(root)
    return nodes, edges
```

`nodes` — every `Value` object in the graph (a Python `set`, so no duplicates).
`edges` — every `(parent, child)` pair as directed connections.
`walk` is a **recursive** function: it calls itself on each parent of `v`.
The `if v not in nodes` guard is essential — without it, a node used twice
(like `a * a`) would cause infinite recursion.

### `draw_dot(root)` — turns nodes and edges into an SVG diagram

```python
def draw_dot(root, rankdir='LR'):
    dot = Digraph(graph_attr={'rankdir': rankdir, 'size': '12,8'})
    nodes, edges = trace(root)

    for n in nodes:
        uid   = str(id(n))
        parts = ([n.label] if n.label else []) + [f'data {n.data:.4f}', f'grad {n.grad:.4f}']
        dot.node(uid, label='{ ' + ' | '.join(parts) + ' }', shape='record')
        if n._op:
            op_uid = uid + n._op
            dot.node(op_uid, label=n._op, shape='circle')
            dot.edge(op_uid, uid)

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot
```

`id(n)` — Python's unique integer ID for every object in memory.
We use it as the node identifier inside graphviz so two different `Value`
objects with the same `data` never get merged.

`shape='record'` — graphviz draws a rectangle with `|` creating vertical dividers.
`shape='circle'` — graphviz draws a small circle (used for the operation bubbles).

Returning the `Digraph` object from the last line of a Jupyter cell renders it as SVG inline.

In [ ]:
# ✏️  type your code here


## Step 9 — Test the visualisation

```python
a = Value(2.0,  label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
f = Value(-2.0, label='f')
e = a * b;  e.label = 'e'
d = e + c;  d.label = 'd'
L = d * f;  L.label = 'L'

draw_dot(L)
```

You should see:
- Rectangles for `a`, `b`, `c`, `f` (leaf inputs) and `e`, `d`, `L` (computed values)
- Small circles labelled `*`, `+`, `*` (the operations)
- Arrows flowing left → right from inputs to outputs

All `grad` values show `0.0000` — correct, we have not run backprop yet.

In [ ]:
# ✏️  type your code here


## The chain rule — what it means and why it works

### Intuition

Imagine two cogs connected together.
Turning cog A by 1 tooth turns cog B by 2 teeth (ratio 2).
Turning cog B by 1 tooth turns cog C by 3 teeth (ratio 3).

**Question:** if I turn cog A by 1 tooth, how much does cog C move?
**Answer:** 2 × 3 = 6 teeth.

You multiplied the ratios along the chain. That is the chain rule.

Formally: if `L = f(g(x))`, then `dL/dx = (dL/dg) × (dg/dx)`.

### In our computation graph

Every path from `L` back to an input `x` passes through a chain of operations.
At each operation we know the **local derivative** (the "ratio" for that one step).
Multiplying all the local derivatives along a path gives us `dL/dx`.

### The `+=` rule

If there are **two paths** from `L` to `x` (e.g. `x` is used twice), both paths
contribute and we must **add** them — not overwrite. This is called the **multivariate
chain rule** and it is why `_backward` always uses `self.grad += ...` not `self.grad = ...`.

### Local derivatives for each operation (derive on paper before moving on)

| Operation | What it computes | Local derivative |
|-----------|-----------------|-----------------|
| `z = a + b` | sum | `∂z/∂a = 1`, `∂z/∂b = 1` |
| `z = a * b` | product | `∂z/∂a = b`, `∂z/∂b = a` |
| `z = aⁿ` | power | `∂z/∂a = n · aⁿ⁻¹` |
| `z = tanh(a)` | hyperbolic tangent | `∂z/∂a = 1 − tanh²(a)` |
| `z = relu(a)` | max(0, a) | `∂z/∂a = 1` if `a > 0`, else `0` |
| `z = exp(a)` | eˣ | `∂z/∂a = exp(a)` (its own derivative) |
| `z = log(a)` | natural log | `∂z/∂a = 1/a` |

For addition: adding a tiny ε to `a` changes `z` by exactly ε regardless of `b`. Derivative = 1.
For multiplication: adding ε to `a` changes `a*b` by `ε*b`. Derivative = `b`. Intuitive.

## Step 10 — Adding `_backward` to `__add__` and `__mul__`

Now we rewrite `__add__` and `__mul__` with gradient functions attached.

A **closure** is a function defined inside another function that captures variables
from the outer scope. When `_backward()` runs later (possibly much later), it still
has access to `self`, `other`, and `out` from the moment the operation happened.

### `__add__` with `_backward`:

```python
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out   = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += out.grad   # local derivative = 1, so just pass upstream grad through
            other.grad += out.grad
        out._backward = _backward
        return out
```

`out.grad` is the gradient flowing in from downstream (the upstream gradient in chain rule terms).
Local derivative of addition is 1 for both inputs, so we multiply by 1 — i.e., pass unchanged.

### `__mul__` with `_backward`:

```python
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out   = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad   # ∂(a*b)/∂a = b
            other.grad += self.data  * out.grad   # ∂(a*b)/∂b = a
        out._backward = _backward
        return out
```

For multiplication, each input's gradient = the *other* input's value × upstream gradient.

**Important:** `out._backward = _backward` installs the function on the output node.
The `lambda: None` placeholder we set in `__init__` gets replaced here.

In [ ]:
# ✏️  type your code here


## Step 11 — `_backward` for `__pow__`, `tanh`, `relu`, `exp`, `log`

Same pattern every time: compute the forward value, save anything needed for the
gradient, define `_backward`, install it.

### `__pow__`:
```python
    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        out = Value(self.data**exp, (self,), f'**{exp}')
        def _backward():
            self.grad += exp * (self.data**(exp - 1)) * out.grad
        out._backward = _backward
        return out
```
Power rule from calculus: `d(xⁿ)/dx = n · xⁿ⁻¹`.
Bring the exponent down, reduce it by 1, multiply by upstream gradient.

### `tanh`:
```python
    def tanh(self):
        t   = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1.0 - t**2) * out.grad
        out._backward = _backward
        return out
```
`t` is computed once outside `_backward` and captured by the closure.
This avoids calling `math.tanh` again during the backward pass.
The derivative identity is: `d(tanh(x))/dx = 1 - tanh(x)²`.

### `relu`:
```python
    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            self.grad += float(out.data > 0) * out.grad
        out._backward = _backward
        return out
```
`out.data > 0` is a Python boolean (`True` or `False`).
`float(True) = 1.0`, `float(False) = 0.0` — this is the gradient gate of relu.

### `exp`:
```python
    def exp(self):
        e   = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out
```
`eˣ` is its own derivative — one of the most remarkable facts in calculus.

### `log`:
```python
    def log(self):
        assert self.data > 0, f'log({self.data}) undefined'
        out = Value(math.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out
```

In [ ]:
# ✏️  type your code here


## Step 12 — The `backward()` method: topological sort + gradient sweep

This is the method you call on the output node (e.g. `loss.backward()`).
It does the full backward pass in two phases.

```python
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
```

### Phase 1: Topological sort

We need to call `_backward` on every node, but in the right order.
A node's `_backward` reads `out.grad` (its output's gradient) — so that output must
already have its gradient filled in before we process the node.

**Topological order** = every node appears *after* all nodes that consume it.

`build` is a recursive post-order DFS (depth-first search):
1. Visit all of `v`'s children first (by calling `build` on each)
2. Then append `v` itself to `topo`

This guarantees that by the time `v` appears in `topo`, all its consumers have
already been appended before it.

`reversed(topo)` therefore walks from the output (`self`) back to the inputs —
exactly the order we need.

### Phase 2: Gradient sweep

`self.grad = 1.0` — seeds the root node with gradient 1.
Interpretation: `∂L/∂L = 1` — the output changes by 1 for every 1-unit change in itself.

`node._backward()` — for each node in reverse topological order, fire its gradient function.
Each call reads `node.grad` (already filled by a previous call) and writes into its parents.

In [ ]:
# ✏️  type your code here


## Step 13 — Complete `Value` class + test backprop

In the cell below, type the **complete updated `Value` class** with everything:
`__init__`, `__repr__`, all ops with `_backward` closures, `backward()`,
and the convenience methods from Step 6.

Then test with Karpathy's classic single-neuron example.
The value of `b` is chosen so `tanh(n) = 0.7071` (i.e. `tanh(n)² = 0.5`).

```python
x1 = Value(2.0,   label='x1')
x2 = Value(0.0,   label='x2')
w1 = Value(-3.0,  label='w1')
w2 = Value(1.0,   label='w2')
b  = Value(6.8813735870195432, label='b')

x1w1   = x1 * w1;     x1w1.label   = 'x1*w1'
x2w2   = x2 * w2;     x2w2.label   = 'x2*w2'
preact = x1w1 + x2w2; preact.label = 'preact'
n      = preact + b;  n.label      = 'n'
o      = n.tanh();    o.label      = 'o'

o.backward()

for v in [o, n, b, preact, x1w1, x2w2, x1, w1, x2, w2]:
    print(f'{v.label:12s}  grad = {v.grad:+.6f}')
```

**Trace through the math manually first:**
- `o.grad = 1.0` — always (dL/dL = 1)
- `n.grad = 1 - tanh(n)² = 1 - 0.5 = 0.5` — tanh derivative
- `w1.grad = x1.data × n.grad = 2 × 0.5 = 1.0` — from the `*` backward
- `x1.grad = w1.data × n.grad = -3 × 0.5 = -1.5`
- `w2.grad = x2.data × n.grad = 0 × 0.5 = 0.0`
- `x2.grad = w2.data × n.grad = 1 × 0.5 = 0.5`

In [ ]:
# ✏️  type your code here


In [ ]:
# test: run o.backward() and print gradients


## Step 14 — Visualise the graph with gradients

After running `o.backward()` in the previous cell, type:

```python
draw_dot(o)
```

Now the `grad` column in each box shows a real value instead of `0.0000`.
You can visually trace how `1.0` at the output node flows backwards through the graph,
getting split, scaled, and accumulated as it reaches `x1`, `w1`, `x2`, `w2`, and `b`.

In [ ]:
# ✏️  type your code here


## Step 15 — Cross-check with PyTorch

Your `pytorch_revision` kernel has PyTorch available. This cell runs the same
computation in PyTorch and compares every gradient to ours.

```python
import torch

x1t = torch.tensor(2.0,  dtype=torch.float64, requires_grad=True)
x2t = torch.tensor(0.0,  dtype=torch.float64, requires_grad=True)
w1t = torch.tensor(-3.0, dtype=torch.float64, requires_grad=True)
w2t = torch.tensor(1.0,  dtype=torch.float64, requires_grad=True)
bt  = torch.tensor(6.8813735870195432, dtype=torch.float64, requires_grad=True)

ot = torch.tanh(x1t*w1t + x2t*w2t + bt)
ot.backward()

pairs = [(x1t,x1,'x1'),(x2t,x2,'x2'),(w1t,w1,'w1'),(w2t,w2,'w2'),(bt,b,'b')]
for pt, ours, name in pairs:
    ok = '✓' if abs(pt.grad.item() - ours.grad) < 1e-9 else '✗'
    print(f'{name}: pytorch={pt.grad.item():+.6f}  ours={ours.grad:+.6f}  {ok}')
```

`requires_grad=True` — tells PyTorch to track this tensor for autograd.
`dtype=torch.float64` — 64-bit floats so the comparison with our engine is exact.
`.item()` — converts a 1-element tensor to a plain Python float.

All five should show `✓`. If any show `✗`, find the `_backward` where the math is wrong.

In [ ]:
# ✏️  type your code here


## What is a neural network?

### The biological inspiration (very loose)

A neuron in your brain receives electrical signals from many other neurons.
If the combined signal is strong enough, it "fires" — sending a signal onwards.
Each connection has a strength (stronger connections contribute more).

### The mathematical model

We model this as:

```
output = activation( w₁x₁ + w₂x₂ + ... + wₙxₙ + b )
```

- `x₁, x₂, ..., xₙ` — inputs (numbers)
- `w₁, w₂, ..., wₙ` — weights (how important each input is)
- `b` — bias (shifts the threshold up or down)
- `activation` — a non-linear function like `tanh` or `relu`

Without the activation function, stacking layers gives you nothing — a stack of
linear functions is still just one linear function.
The non-linearity is what makes deep networks powerful.

### A layer = many neurons reading the same input

A **layer** is just several neurons applied to the same input, producing several outputs.

### An MLP = several layers in sequence

The outputs of layer 1 become the inputs of layer 2, and so on.
This lets the network learn increasingly abstract features.

### What does "training" mean?

The weights `w` and biases `b` start as random numbers.
Training adjusts them so the network's outputs match the desired outputs.
The adjustment rule is: move each weight in the direction that reduces the loss.
That direction is given by the negative gradient — which our autograd engine computes.

## Step 16 — `Module`, `Neuron`, `Layer`, `MLP`

### `Module` — base class for all network components

```python
class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0
    def parameters(self):
        return []
```

`zero_grad()` resets every parameter's `.grad` to 0.
**Must be called before every `backward()`** because `_backward` uses `+=`.
Without zeroing, gradients from epoch 1 add on top of epoch 2, the numbers explode,
and the model diverges.

`parameters()` returns an empty list by default. Subclasses override it.

### `Neuron`

```python
class Neuron(Module):
    def __init__(self, n_in, activation='tanh'):
        self.w          = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b          = Value(0.0)
        self.activation = activation

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
        if self.activation == 'tanh':  return act.tanh()
        if self.activation == 'relu':  return act.relu()
        return act

    def parameters(self):
        return self.w + [self.b]
```

`self.w = [Value(random.uniform(-1,1)) ...]` — one weight per input, randomly set in [-1, 1].
Random init breaks symmetry: if all weights were identical, all neurons would learn identically.

`sum(..., Value(0.0))` — `sum` takes an optional start value. We use `Value(0.0)` so the
result is a `Value` object (not a plain float), keeping it inside the computation graph.

`zip(self.w, x)` — pairs each weight with its corresponding input. Like a zipper.

`__call__` — defining this makes `neuron(x)` work like a function call.

### `Layer`

```python
class Layer(Module):
    def __init__(self, n_in, n_out, **kwargs):
        self.neurons = [Neuron(n_in, **kwargs) for _ in range(n_out)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]
```

`**kwargs` — the `**` unpacks a dictionary into keyword arguments. It passes extra
arguments like `activation='relu'` directly down to each `Neuron` without us having
to name them explicitly.

`outs[0] if len(outs) == 1 else outs` — if the layer has only 1 neuron (like the
final output layer), return the single value directly, not a list of one item.

### `MLP`

```python
class MLP(Module):
    def __init__(self, n_in, layer_sizes):
        dims = [n_in] + layer_sizes
        self.layers = [
            Layer(dims[i], dims[i+1],
                  activation='tanh' if i < len(layer_sizes) - 1 else 'linear')
            for i in range(len(layer_sizes))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
```

`dims = [n_in] + layer_sizes` — for `MLP(2, [4, 4, 1])`, `dims = [2, 4, 4, 1]`.
Layer 0: 2→4. Layer 1: 4→4. Layer 2: 4→1.

Hidden layers use `tanh` (non-linear). The final layer is `'linear'` — no activation —
so the output is a raw number, not squashed into (-1, +1). This lets the model reach
any target value during training.

**Type all four classes in the cell below.**

In [ ]:
# ✏️  type your code here


## Step 17 — Test the MLP

```python
random.seed(0)
net = MLP(2, [4, 4, 1])
print(f'Parameters: {len(net.parameters())}')

out = net([1.0, -2.0])
print(f'Output: {out}')
```

**Count the parameters yourself:**
- Layer 0 (2→4): each of 4 neurons has 2 weights + 1 bias = 3 params. 4 × 3 = 12
- Layer 1 (4→4): each of 4 neurons has 4 weights + 1 bias = 5 params. 4 × 5 = 20
- Layer 2 (4→1): 1 neuron, 4 weights + 1 bias = 5 params

Total: 12 + 20 + 5 = **37 parameters**.

In [ ]:
# ✏️  type your code here


## What is gradient descent? What is a loss function?

### Loss function

A **loss function** measures how wrong the model is — one single number.
Lower loss = better model. We want to minimise it.

**Mean Squared Error (MSE)** — what we will use:
```
loss = (1/N) × Σ (prediction - target)²
```
For each example, square the difference between what the model predicted and what
we wanted. Average over all examples. Always ≥ 0. Equals 0 only if perfect.

### Gradient descent

The gradient `∂loss/∂w` tells us: if we increase weight `w` by a tiny amount,
does the loss go up or down?

If `∂loss/∂w > 0`: increasing `w` increases loss → we should *decrease* `w`.
If `∂loss/∂w < 0`: increasing `w` decreases loss → we should *increase* `w`.

The update rule:
```
w ← w - lr × ∂loss/∂w
```

`lr` is the **learning rate** — how big a step to take. Too large: overshoot and diverge.
Too small: training takes forever. Typically something between 0.001 and 0.1.

### Why not just solve it directly?

For a loss surface with millions of parameters, there is no closed-form solution.
Gradient descent navigates the surface step by step, always moving downhill.

### The four-line training loop

Every neural network ever trained runs these four operations every step:

```
1. forward   — compute predictions and loss
2. zero_grad — reset all .grad to 0
3. backward  — fill in all .grad with ∂loss/∂param
4. step      — param.data -= lr × param.grad
```

The order of 2 and 3 matters: zero *before* backward, not after.

## Step 18 — Training loop: learning XOR

**XOR problem:** given two bits, output 1 if they differ, -1 if they match.
It is the simplest problem that is not linearly separable — you cannot solve it
with a straight line. A two-layer network handles it easily.

```python
random.seed(42)
model = MLP(2, [8, 8, 1])

X = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
y = [-1.0,         1.0,        1.0,        -1.0]

losses = []
for epoch in range(400):
    # 1. forward pass
    preds = [model(xi) for xi in X]
    loss  = sum((p - yi)**2 for p, yi in zip(preds, y)) * (1.0 / len(y))

    # 2. zero gradients
    model.zero_grad()

    # 3. backward pass
    loss.backward()

    # 4. SGD step
    lr = 0.05
    for p in model.parameters():
        p.data -= lr * p.grad

    losses.append(loss.data)
    if epoch % 50 == 0:
        print(f'epoch {epoch:3d}  loss = {loss.data:.6f}')

print()
print('Final predictions:')
for xi, yi in zip(X, y):
    pred = model(xi)
    print(f'  {xi}  target={yi:+.0f}  pred={pred.data:+.4f}')
```

**Line-by-line:**

`preds = [model(xi) for xi in X]` — run all 4 inputs through the model.
Each `model(xi)` call does a full forward pass and builds a computation graph.

`loss = sum((p - yi)**2 ...) * (1.0 / len(y))` — MSE loss.
`(p - yi)` works because `Value.__sub__` is defined.
`**2` works because `Value.__pow__` is defined.
`sum(...) * (1/4)` works because `Value.__mul__` is defined.
The entire loss is a `Value` connected to all 37 parameters through the graph.

`model.zero_grad()` — reset every parameter's `.grad` to 0.
If you skip this, old gradients accumulate and the model diverges.

`loss.backward()` — topological sort + gradient sweep across the entire graph.
After this, every `p.grad` holds `∂loss/∂p`.

`p.data -= lr * p.grad` — plain Python arithmetic. `p.grad` is just a float.
We directly modify `p.data` — this is the weight update.

In [ ]:
# ✏️  type your code here


## Step 19 — Plot the training loss

No matplotlib needed — this prints an ASCII chart with a log scale:

```python
import math as _m

width  = 60
mn, mx = min(losses), max(losses)
lo, hi = _m.log(mn + 1e-9), _m.log(mx + 1e-9)
step   = max(1, len(losses) // 30)

print(f'Loss  {losses[0]:.5f}  →  {losses[-1]:.5f}  (log scale)')
print('-' * (width + 16))
for i in range(0, len(losses), step):
    frac = (_m.log(losses[i] + 1e-9) - lo) / (hi - lo) if hi > lo else 0
    bar  = '█' * int(frac * width)
    print(f'ep {i:3d} | {bar:<{width}} {losses[i]:.5f}')
```

**Why log scale?** The loss drops fast at the beginning and slowly at the end.
On a linear scale the early drama is invisible. Log scale makes both phases visible.

`+ 1e-9` — `1e-9` means `0.000000001`. Added to prevent `log(0)` if loss hits zero.

`{bar:<{width}}` — the `<` left-aligns the bar string inside a field `width` characters wide,
so all the loss numbers line up on the right.

In [ ]:
# ✏️  type your code here


## You built it.

Start to finish, in pure Python, with no deep learning libraries:

| What you built | What it does |
|---------------|-------------|
| `Value.data` / `_prev` / `_op` | Records the computation graph during the forward pass |
| `Value._backward` closures | Implements the chain rule for each operation |
| `Value.backward()` | Topological sort + gradient sweep across the full graph |
| `Neuron`, `Layer`, `MLP` | Neural network built purely from `Value` arithmetic |
| Training loop | Forward → zero → backward → SGD, repeated until the loss drops |

**The connection to PyTorch:**
PyTorch's `Tensor` is exactly a `Value`, but extended to work on multi-dimensional arrays
(matrices) instead of scalars, and accelerated on GPU. The backward mechanics — closures,
topological sort, chain rule — are identical to what you just wrote.

You now have the mental model to understand what happens inside `loss.backward()`
in any PyTorch tutorial you encounter.